# 04 — Análise de erros do Random Forest
Modelo: `train_random_forest_20261103T170044Z_3b8896f7` · threshold = 0,50.
Análise exploratória apenas da validação de desenvolvimento (1.600 registros, 35 falhas). **Não usa TEST**. Para executar, baixe o artefato `validation-balancing-results` da [execução de referência](https://github.com/Mateusssssssssss/Predictive-Maintenance/actions/runs/41679733671) e extraia na raiz do projeto; o ZIP contém `metrics/<run_id>/validation_predictions.csv`. Não confundir com a rodada posterior de threshold reduzido.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score

RUN = 'train_random_forest_20261103T170044Z_3b8896f7'
paths = [Path('reports/metrics') / RUN / 'validation_predictions.csv', Path('metrics') / RUN / 'validation_predictions.csv']
source = next((p for p in paths if p.is_file()), None)
if source is None: raise FileNotFoundError('Extraia o artefato da execução 41679733671 na raiz do projeto.')
df = pd.read_csv(source)
assert len(df)==1600 and df.y_true.sum()==35
assert np.array_equal(df.prediction, (df.score>=.5).astype(int))
df['resultado']=np.select([df.y_true.eq(1)&df.prediction.eq(0),df.y_true.eq(1)&df.prediction.eq(1),df.y_true.eq(0)&df.prediction.eq(1)],['FN','TP','FP'],default='TN')
print(df.resultado.value_counts()); print('Average Precision:', average_precision_score(df.y_true,df.score))

## Nove falsos negativos


In [ ]:
features=['Air temperature [K]','Process temperature [K]','Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
fn=df[df.resultado=='FN'].sort_values('score',ascending=False)
display(fn[['UDI','Type','score']+features])
assert len(fn)==9

## Matriz de confusão e scores


In [ ]:
cm=confusion_matrix(df.y_true,df.prediction,labels=[0,1]); fig,ax=plt.subplots(figsize=(5,4));ax.imshow(cm,cmap='Blues')
for (i,j),n in np.ndenumerate(cm):ax.text(j,i,str(n),ha='center',va='center',fontsize=16)
ax.set(xlabel='Predito',ylabel='Real',title='Matriz de confusão');plt.show()
fig,ax=plt.subplots(figsize=(9,4))
for k in ['TN','TP','FN']:
    series=df.loc[df.resultado==k,'score'].clip(lower=1e-6)
    ax.hist(series,bins=np.geomspace(1e-6,1,45),alpha=.5,label=f'{k}: {len(series)}')
ax.set_xscale('log');ax.axvline(.5,color='black',ls='--');ax.set(xlabel='Score (escala log)',ylabel='Registros');ax.legend();plt.show()

## Curva Precision–Recall e sensores
O conjunto de validação já foi consultado. Não otimizar o threshold aqui e reutilizar a mesma validação como resultado independente.

In [ ]:
precision,recall,_=precision_recall_curve(df.y_true,df.score)
fig,ax=plt.subplots(figsize=(7,4));ax.plot(recall,precision);ax.axvline(.9,color='black',ls='--');ax.axhline(.8,color='gray',ls='--');ax.scatter([26/35],[1]);ax.set(xlabel='Recall',ylabel='Precision',title='Curva Precision–Recall');plt.show()
tp=df[df.resultado=='TP'];fig,axes=plt.subplots(1,5,figsize=(17,4),constrained_layout=True)
for ax,feature in zip(axes,features):
    ax.boxplot([fn[feature],tp[feature]],tick_labels=['FN','TP']);ax.set_title(feature.replace(' [','\n['),fontsize=10)
plt.show()
summary=df[df.y_true==1].groupby('Type').resultado.value_counts().unstack(fill_value=0)
display(summary)
summary.plot(kind='bar',stacked=True,title='Falhas por tipo de produto');plt.show()

## Interpretação e limites
Os nove FN têm scores entre 0,0071 e 0,0304. A mediana FN para torque é 38,9 Nm (TP: 60,8 Nm); para rotação 1.524 rpm (TP: 1.349 rpm); desgaste 214 min (TP: 146 min). Essas diferenças são descritivas e não estabelecem causalidade. Com apenas 35 positivos, um único caso altera o recall em cerca de 2,86 pontos percentuais. Evite retunar com base nestes registros e depois declarar avaliação independente. O UDI não é tempo comprovado.

## Diagnóstico adicional: modos de falha (somente investigação)

O dataset inclui colunas auxiliares `TWF`, `HDF`, `PWF`, `OSF` e `RNF`. **Elas jamais devem ser usadas como features preditivas**, porque descrevem modos de falha; são usadas abaixo apenas para auditar os erros, após a predição.

Hipótese encontrada nesta validação: 7 dos 9 FN têm `TWF=1`; nenhum dos 26 TP tem `TWF=1`. Esse padrão não determina causalidade e não assegura generalização. Também há casos cuja classe principal difere da união das flags auxiliares.

In [ ]:
raw = pd.read_csv('manutencao.csv').sort_values('UDI')
assert raw.UDI.is_unique and len(raw) == 10000
modes = ['TWF','HDF','PWF','OSF','RNF']
observed = raw.set_index('UDI').loc[df.UDI, modes].reset_index()
observed['resultado'] = df.resultado.to_numpy()
counts = observed.groupby('resultado')[modes].sum()
display(counts.loc[[k for k in ['FN','TP'] if k in counts.index]])
fig, ax = plt.subplots(figsize=(8,4))
counts.loc[['FN','TP']].T.plot(kind='bar', ax=ax, rot=0)
ax.set(title='Modos auxiliares entre falhas não detectadas e detectadas',
       ylabel='Quantidade', xlabel='Indicador auxiliar')
plt.show()
assert int(counts.loc['FN','TWF']) == 7
assert int(counts.loc['TP','TWF']) == 0

## Semelhança entre falsos negativos e registros do treinamento

Usamos **somente os 4.800 registros FIT_SEARCH** para buscar os 20 vizinhos mais próximos de cada falha na validação. A distância é euclidiana nas cinco variáveis numéricas padronizadas **com média e desvio estimados no FIT_SEARCH**. A comparação é restrita ao mesmo `Type`.

A taxa de falhas dos 20 vizinhos é uma descrição de densidade local, **não uma nova previsão** nem uma estimativa calibrada de risco. O procedimento tem limitações devido à métrica de distância e às relações não lineares.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

numeric = ['Air temperature [K]','Process temperature [K]',
           'Rotational speed [rpm]','Torque [Nm]','Tool wear [min]']
train = raw[raw.UDI <= 4800].copy()
scaler = StandardScaler().fit(train[numeric])
train_scaled = scaler.transform(train[numeric])
validation_scaled = scaler.transform(df[numeric])
output = []
for type_value in train.Type.unique():
    train_mask = train.Type.eq(type_value).to_numpy()
    val_mask = df.Type.eq(type_value).to_numpy()
    if not val_mask.any():
        continue
    local = train.loc[train_mask].reset_index(drop=True)
    nearest = NearestNeighbors(n_neighbors=20).fit(train_scaled[train_mask])
    distances, indices = nearest.kneighbors(validation_scaled[val_mask])
    subset = df.loc[val_mask].reset_index(drop=True)
    for i in range(len(subset)):
        neighbors = local.iloc[indices[i]]
        output.append({
            'UDI': int(subset.loc[i,'UDI']),
            'resultado': subset.loc[i,'resultado'],
            'Type': type_value,
            'falhas_entre_20_vizinhos': int(neighbors['Machine failure'].sum()),
            'distancia_20o_vizinho': float(distances[i,-1])
        })
nearest_results = pd.DataFrame(output)
display(nearest_results[nearest_results.resultado=='FN'].sort_values('UDI'))
display(nearest_results[nearest_results.resultado.isin(['FN','TP'])]
        .groupby('resultado')['falhas_entre_20_vizinhos'].describe())

In [ ]:
fig, ax = plt.subplots(figsize=(8,4))
samples = [nearest_results.loc[nearest_results.resultado==key,
                               'falhas_entre_20_vizinhos'] for key in ['FN','TP']]
ax.boxplot(samples, tick_labels=['FN (9)','TP (26)'])
ax.set(title='Falhas entre os 20 vizinhos mais próximos no FIT_SEARCH',
       ylabel='Quantidade de vizinhos rotulados como falha')
plt.show()
assert nearest_results.query("resultado == 'FN'").falhas_entre_20_vizinhos.median() == 1
assert nearest_results.query("resultado == 'TP'").falhas_entre_20_vizinhos.median() == 7

## Recomendações para experimentos futuros

1. Investigar por que o mecanismo TWF não está sendo capturado pelas features disponíveis, considerando a semântica do target e discrepâncias entre os rótulos auxiliares.
2. Comparar indicadores derivados de desgaste e carga **sem incluir as colunas TWF/HDF/PWF/OSF/RNF como preditores**.
3. Reavaliar com dados independentes por equipamento/tempo verdadeiros; `UDI` não é tempo comprovado.
4. Não escolher novas features ou limiares usando este mesmo bloco de validação e depois anunciar uma melhoria independente.